# Quiz 2

### Imports

In [10]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
from scipy import stats

### Datasets

In [11]:
df_1 = pd.read_csv('./datasets/homework_2.1.csv')
df_2 = pd.read_csv('./datasets/homework_2.2.csv')

### Question 1

In [12]:
df_1.head()

,time,G1,G2,G3
0,0,0.882026,1.441575,0.065409
1,1,0.210079,-0.163880,0.140310
2,2,0.509369,-0.115242,0.819830
3,3,1.150447,1.014698,0.607632
4,4,0.973779,-0.046562,0.610066


In [13]:
df_long = df_1.melt(id_vars='time', value_vars=['G1', 'G2', 'G3'], var_name='group', value_name='outcome')

df_long['time_dm'] = df_long['time'] - df_long.groupby('group')['time'].transform('mean')
df_long['outcome_dm'] = df_long['outcome'] - df_long.groupby('group')['outcome'].transform('mean')

slope = sm.OLS(df_long['outcome_dm'], df_long[['time_dm']]).fit().params['time_dm']
print(f"Common slope: {slope:.6f}")

print('-----')

group_means = df_long.groupby('group')[['time', 'outcome']].mean()
group_means['fixed_effect'] = group_means['outcome'] - slope * group_means['time']

print(group_means[['fixed_effect']])

Common slope: 0.009017
-----
       fixed_effect
group              
G1         0.078552
G2         0.589654
G3         0.269032


### Question 3

In [14]:
df_2.head()

,X,Y,Z
0,0,1.182435,-0.725820
1,0,2.714474,0.563476
2,0,0.077612,-0.435632
3,0,-0.154449,-0.104553
4,0,22.298992,-2.321273


In [15]:
treated_mean = df_2[df_2['X'] == 1]['Y'].mean()
untreated_mean = df_2[df_2['X'] == 0]['Y'].mean()

simple_effect = treated_mean - untreated_mean
print(f"{simple_effect:.3f}")

2.921


### Problem 4

In [19]:
n_bootstrap = 10000
bootstrap_effects = []
boostrap_var_list = []

for i in range(10):
    for j in range(n_bootstrap):
        sample = df_2.sample(n=len(df_2), replace=True)
        
        treated_mean   = sample[sample['X'] == 1]['Y'].mean()
        untreated_mean = sample[sample['X'] == 0]['Y'].mean()
        
        bootstrap_effects.append(treated_mean - untreated_mean)

        if j % 100 == 0:
            print(f"{i+1}: {j}/{n_bootstrap}", end='\r', flush=True)

    boostrap_var_list.append(np.var(bootstrap_effects))

print(f"\n{np.mean(boostrap_var_list):.4f}")

10: 9900/10000
0.0320


### Problem 5

In [21]:
np.random.seed(42)
n_bootstrap = 100000
bootstrap_effects = []

for i in range(n_bootstrap):
    sample = df_2.sample(n=len(df_2), replace=True)
    
    X = sm.add_constant(sample['X'])
    y = sample['Y']
    
    model = sm.OLS(y, X).fit()
    bootstrap_effects.append(model.params['X'])

    if i % 100 == 0:
        print(f"{i}/{n_bootstrap}", end='\r', flush=True)

skewness = stats.skew(bootstrap_effects)
print(f"\n{skewness:.4f}")

99900/100000
0.0438
